# EconReview on Google Colab

Run the Economics model and backend in this notebook. Mark an answer against a bank question, select a missed marking point, and answer the tutor's probe.

In **Runtime → Change runtime type**, select a GPU, then run the cells from top to bottom. An L4 (24 GB) may fit and needs a rehearsal. An A100 or a GPU with more memory gives the current BF16 model more room. The T4 (16 GB) is too small.

Leave `MODEL_FOLDER` blank to download the pinned 19.3 GB checkpoint directly to Colab's temporary disk. To reuse an exact copy from Google Drive, set `MODEL_FOLDER` to its snapshot folder under `/content/drive/MyDrive`. Your answer data stays in the runtime and resets when Colab deletes it.

In [ ]:
MODEL_FOLDER = ''  # e.g. '/content/drive/MyDrive/Models/Qwen3.5-9B-Base'
PROJECT_ZIP = ''   # optional, e.g. '/content/econreview-comp4431.zip'

import json
import os
import secrets
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import requests
import torch

if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported(including_emulation=False):
    raise RuntimeError('Select a CUDA GPU with native BF16 support in the Colab runtime settings.')

workspace = Path('/content')
project = workspace / 'econreview-comp4431'
if not project.exists():
    if PROJECT_ZIP:
        with zipfile.ZipFile(PROJECT_ZIP) as archive:
            archive.extractall(workspace)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/benkuan08/econreview-comp4431.git', str(project)], check=True)

run = project / 'training_output/shared-economics'
adapter_dir = run / 'adapter'
source = json.loads((adapter_dir / 'source_model.json').read_text())
shards = json.loads((run / 'runtime/weight_manifest.json').read_text())
weight_bytes = sum(item['size'] for item in shards.values())
free, total = torch.cuda.mem_get_info()
print(f'{torch.cuda.get_device_name()}: {free / 2**30:.1f} of {total / 2**30:.1f} GiB free')
if free <= weight_bytes:
    raise RuntimeError('This runtime lacks room for the BF16 weights alone. Select a GPU with more free VRAM.')

venv = workspace / 'econreview-venv'
python = str(venv / 'bin/python')
if not Path(python).is_file():
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(venv)], check=True)
subprocess.run([python, '-m', 'pip', 'install', '-q', '-r', 'backend/requirements.txt', '-r', 'training/requirements-inference.txt'], cwd=project, check=True)

In [ ]:
# Restore the trained adapter and verify the exact base weights.
if MODEL_FOLDER.startswith('/content/drive/'):
    from google.colab import drive
    drive.mount('/content/drive')

model_dir = Path(MODEL_FOLDER or '/content/models/Qwen3.5-9B-Base').expanduser().resolve()
if not MODEL_FOLDER:
    model_dir.mkdir(parents=True, exist_ok=True)
    remaining = sum(item['size'] for name, item in shards.items() if not (model_dir / name).is_file())
    if shutil.disk_usage(model_dir).free < remaining:
        raise RuntimeError('Free disk space in the Colab runtime for the remaining model weights.')
    download = 'from huggingface_hub import snapshot_download; import sys; snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], local_dir=sys.argv[3])'
    subprocess.run([python, '-c', download, source['id'], source['sha'], str(model_dir)], check=True)

subprocess.run([python, 'scripts/download_base_model.py', '--verify-dir', str(model_dir)], cwd=project, check=True)
subprocess.run([python, 'scripts/download_adapter.py'], cwd=project, check=True)

In [ ]:
def stop_services():
    for process in reversed(globals().get('services', [])):
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()

stop_services()
services = []
data_dir = workspace / 'econreview-data'
data_dir.mkdir(exist_ok=True)
env = dict(os.environ,
    BACKEND_MODEL_ID=source['id'], BACKEND_MODEL_REVISION=source['sha'],
    BACKEND_MODEL_DIR=str(model_dir), BACKEND_ADAPTER=str(adapter_dir),
    BACKEND_MODEL_TOKEN=secrets.token_hex(32), BACKEND_MODEL_URL='http://127.0.0.1:8100',
    BACKEND_DATABASE_URL=f'sqlite:///{data_dir / "backend.db"}',
    BACKEND_STORAGE_DIR=str(data_dir / 'files'), BACKEND_WORKER_ENABLED='true',
    HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1',
)

def start_service(module, port, ready_path, timeout):
    log = data_dir / f'{module}.log'
    with log.open('w') as stream:
        process = subprocess.Popen([python, '-m', 'uvicorn', f'{module}:app', '--host', '127.0.0.1', '--port', str(port)],
                                   cwd=project, env=env, stdout=stream, stderr=subprocess.STDOUT)
    services.append(process)
    deadline = time.monotonic() + timeout
    while process.poll() is None and time.monotonic() < deadline:
        try:
            response = requests.get(f'http://127.0.0.1:{port}{ready_path}', timeout=2)
            if response.ok:
                print(f'{module} ready')
                return
        except requests.RequestException:
            pass
        time.sleep(2)
    raise RuntimeError(f'{module} failed to start. Last log lines:\n' + '\n'.join(log.read_text().splitlines()[-20:]))

try:
    start_service('backend.gpu_service', 8100, '/openapi.json', 600)
    start_service('backend.app', 8000, '/health', 60)
except BaseException:
    stop_services()
    raise

In [ ]:
QUESTION_ID = 'pearson-igcse-4ec1-01r-2025-06-q1h'
ANSWER = 'The supply of labour increases.'
SCAN_PATH = ''  # optional JPEG, PNG or PDF uploaded through the Colab Files panel

from IPython.display import Image, display

def api(path, method='GET', **kwargs):
    response = requests.request(method, f'http://127.0.0.1:8000/api/v1{path}', timeout=60, **kwargs)
    if not response.ok:
        raise RuntimeError(f'{response.status_code} from {path}: {response.text}')
    return response.json()

def wait_for_job(job_id):
    deadline = time.monotonic() + 900
    while time.monotonic() < deadline:
        job = api(f'/jobs/{job_id}')
        if job['status'] == 'succeeded':
            return job['result']
        if job['status'] == 'failed':
            raise RuntimeError(job['error'] or 'The backend did not complete this step.')
        time.sleep(2)
    raise TimeoutError(f'Job {job_id} is still running. Check the backend logs in /content/econreview-data.')

questions = api('/questions', params={'search': QUESTION_ID})
question = next((item for item in questions if item['q_id'] == QUESTION_ID), None)
if question is None:
    raise ValueError('Set QUESTION_ID to an exact question ID from the question bank.')
print(question['question'])

answer_body = {'question_id': question['id'], 'text': ANSWER}
if SCAN_PATH:
    scan_path = Path(SCAN_PATH)
    with scan_path.open('rb') as stream:
        document = api('/documents', 'POST', files={'file': (scan_path.name, stream)}, data={'purpose': 'student_answer'})
    document_id = document['id']
    preview = requests.get(f'http://127.0.0.1:8000/api/v1/documents/{document_id}/pages/0', timeout=60)
    preview.raise_for_status()
    display(Image(data=preview.content, width=500))
    text = input('Type or paste the confirmed answer from the scan: ')
    transcript = api(f'/documents/{document_id}/transcripts', 'POST', json={'text': text, 'confirmed': True, 'regions': []})
    answer_body = {'question_id': question['id'], 'transcript_id': transcript['id']}

submission_id = api('/submissions', 'POST')['id']
answer_id = api(f'/submissions/{submission_id}/answers', 'POST', json=answer_body)['id']
queued = api(f'/answers/{answer_id}/grades', 'POST')
wait_for_job(queued['job_id'])
grade_id = queued['grade_id']
grade = api(f'/grades/{grade_id}')
result = grade['result']
print(f'Grade status: {grade["status"]}')
if result:
    print(f'Provisional mark: {result["score"]}/{result["max_score"]} | {result["feedback"]}')
    for point in result['point_results']:
        print(f'{point["mp_id"]} {point["status"]} | {point["reason"]}')

In [ ]:
if not result or result['needs_human_review']:
    raise RuntimeError('A reliable grade is required before tutoring.')
missed = [point['mp_id'] for point in result['point_results'] if point['status'] != 'met']
if not missed:
    raise ValueError('This answer met all marking points. Try a shorter answer to demo tutoring.')
point_id = input(f'Pick a missed marking point from {missed}, or press Enter for {missed[0]}: ').strip() or missed[0]
started = api(f'/grades/{grade_id}/tutoring-sessions', 'POST', json={'mp_id': point_id})
if started.get('job_id'):
    wait_for_job(started['job_id'])
tutorial_path = f'/tutoring-sessions/{started["tutorial_id"]}'

def show_tutor(session):
    print(f'Focused marking point: {session["focused_mp_id"]}')
    for message in session['messages']:
        if message['kind'] == 'lesson':
            print(message['explanation'])
            if message['worked_example']:
                print(message['worked_example'])
    if session['active_probe']:
        print(session['active_probe']['question'])
    else:
        print(session['stop_reason'] or session['state'])

show_tutor(api(tutorial_path))

In [ ]:
# Rerun this cell for each follow-up probe.
session = api(tutorial_path)
if session['active_probe']:
    reply = input(session['active_probe']['question'] + ' ')
    queued = api(f'{tutorial_path}/probe-replies', 'POST', json={'version': session['version'], 'text': reply})
    wait_for_job(queued['job_id'])
    session = api(tutorial_path)
show_tutor(session)

In [ ]:
# Run after the demo to release GPU memory.
stop_services()